# Introduction

---

This notebook runs Gemini 3 pro and flash with vertex ai provider which supports tool usage. This is used with langchain with structured output.

## Imports

In [ ]:
import sys

sys.path.append("..")

In [ ]:
import os
import yaml
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from itertools import repeat
import json
from tqdm import tqdm
from typing import Dict, Union, List, Optional

In [ ]:
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGSMITH_TRACING"] = "false"
os.environ.pop("LANGCHAIN_API_KEY", None)
os.environ.pop("LANGSMITH_API_KEY", None)

In [ ]:
from src.data_loader.data_loader import get_flatten_radqa_data

from src.utils import get_message_for_llm
from src.few_shot import sample_demonstrations, render_demonstrations

from src.providers.llm_response import get_response_from_llm
from src.providers.llm_langchain_providers import get_gemini_llm_with_structured_op
from src.logging import RadQALLMEvalLogger

In [ ]:
from src.prompts.gemini_prompt import GEMINI_PROMPT

## Data Loading

In [ ]:
DATA_PATH = os.path.abspath("../data/radqa/1.0.0/")
assert os.path.exists(DATA_PATH)

In [ ]:
TRAIN_FILE_PATH = os.path.join(DATA_PATH, "train.json")
DEV_FILE_PATH = os.path.join(DATA_PATH, "dev.json")
TEST_FILE_PATH = os.path.join(DATA_PATH, "test.json")

assert os.path.exists(TRAIN_FILE_PATH)
assert os.path.exists(DEV_FILE_PATH)
assert os.path.exists(TEST_FILE_PATH)

In [ ]:
K_SHOT = 0
DEMO_SEED = 20260719

results_subdir = "results" if K_SHOT == 0 else f"results-{K_SHOT}shot"
RESULTS_DIR = os.path.abspath(f"../{results_subdir}/")
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
LOG_DIR_PATH = os.path.abspath("../logs/")
assert os.path.exists(LOG_DIR_PATH)

In [ ]:
with open("../config.yaml", "r") as fp:
    config_data = yaml.safe_load(fp)

## Logger Setup

In [ ]:
LOG_FILE_PATH = os.path.join(LOG_DIR_PATH, config_data["app"]["log_file_name"])

In [ ]:
logger_instance = RadQALLMEvalLogger(
    app_name=config_data["app"]["app_name"], log_file_path=LOG_FILE_PATH
)
app_logger = logger_instance.get_logger()

In [ ]:
app_logger.info(f"Config Data: {config_data}")

In [ ]:
config_data

## Debug

In [ ]:
# radqa_data_flatten_df = get_flatten_radqa_data(file_path=TRAIN_FILE_PATH)
# print(radqa_data_flatten_df.shape)
# radqa_data_flatten_records = radqa_data_flatten_df.to_dict(orient="records")
# sample_records = radqa_data_flatten_records[:5]
# sample_records

In [ ]:
# sample_records = [
#     {
#         "context": "",
#         "question": "",
#     }
# ]

In [ ]:
# messages_batch = [
#     get_message_for_llm(
#         model_id=config_data["vertex_gcp"]["models"]["gemini"][0]["model_id"],
#         context=item["context"],
#         question=item["question"],
#         system_prompt=GEMINI_PROMPT,
#     )
#     for item in sample_records
# ]

In [ ]:
# messages_batch

In [ ]:
# gemini_llm = get_gemini_llm_with_structured_op(
#     model_id=config_data["vertex_gcp"]["models"]["gemini"][0]["model_id"],
#     model_config=config_data["vertex_gcp"]["models"]["gemini"][0]["model_config"],
# )

In [ ]:
# gemini_llm.invoke(messages_batch[0])

In [ ]:
# response = get_response_from_llm(
#     model_id=config_data["vertex_gcp"]["models"]["gemini"][0]["model_id"],
#     messages=messages_batch[0],
#     llm_with_structured_op=gemini_llm,
#     logger=app_logger,
# )

In [ ]:
# response

In [ ]:
# json.dumps(response)

## Process LLM

In [ ]:
def wrapper(
    data_subset_file_path: str,
    vertex_models: List[Dict[str, Union[str, Dict[str, Union[str, int]]]]],
    result_path: str,
    start_idx: int,
    end_idx: Optional[int],
    demonstrations_block: str = "",
) -> None:
    data_subset_name = os.path.splitext(data_subset_file_path.split("/")[-1])[0]
    print(f"Data Subset Name: {data_subset_name}")
    radqa_data_flatten_df = get_flatten_radqa_data(file_path=data_subset_file_path)
    print(radqa_data_flatten_df.shape)
    if end_idx is None:
        end_idx = radqa_data_flatten_df.shape[0]
    radqa_data_flatten_records = radqa_data_flatten_df.to_dict(orient="records")[
        start_idx:end_idx
    ]
    models = vertex_models["models"]
    for model_family, model_details in models.items():
        system_prompt = GEMINI_PROMPT
        for model_info in model_details:
            model_id = model_info["model_id"]
            model_config = model_info["model_config"]
            messages_batch = [
                get_message_for_llm(
                    model_id=model_id,
                    context=item["context"],
                    question=item["question"],
                    system_prompt=system_prompt,
                    demonstrations_block=demonstrations_block,
                )
                for item in radqa_data_flatten_records
            ]
            print(f"Processing for Model Id: {model_id}")
            gemini_llm_with_structured_op = get_gemini_llm_with_structured_op(
                model_id=model_id,
                model_config=model_config,
            )
            with ThreadPoolExecutor(max_workers=2) as executor:
                results = list(
                    tqdm(
                        executor.map(
                            get_response_from_llm,
                            repeat(model_id),
                            messages_batch,
                            repeat(gemini_llm_with_structured_op),
                            repeat(app_logger),
                        ),
                        total=len(messages_batch),
                    )
                )

            merged_records_to_save = [
                {**input, **response}
                for input, response in zip(radqa_data_flatten_records, results)
            ]
            results_dir = os.path.join(result_path, model_id)
            os.makedirs(results_dir, exist_ok=True)
            with open(
                f"{results_dir}/{data_subset_name}_{start_idx}_{end_idx}.json", "w"
            ) as fp:
                json.dump(merged_records_to_save, fp, indent=4)

### Train Set

In [ ]:
# start_idx = 4700
# end_idx = None
# wrapper(
#     data_subset_file_path=TRAIN_FILE_PATH,
#     vertex_models=config_data["vertex_gcp"],
#     result_path=RESULTS_DIR,
#     start_idx=start_idx,
#     end_idx=end_idx,
# )

### Dev Set

In [ ]:
# start_idx = 600
# end_idx = None
# wrapper(
#     data_subset_file_path=DEV_FILE_PATH,
#     vertex_models=config_data["vertex_gcp"],
#     result_path=RESULTS_DIR,
#     start_idx=start_idx,
#     end_idx=end_idx,
# )

## Test connection

In [ ]:
# start_idx = 0
# end_idx = 1
# wrapper(
#     data_subset_file_path=TEST_FILE_PATH,
#     vertex_models=config_data["vertex_gcp"],
#     result_path=RESULTS_DIR,
#     start_idx=start_idx,
#     end_idx=end_idx,
#     demonstrations_block=demonstrations_block,
# )

### Test Set

Few-shot demos are drawn from the train split once (fixed seed) and reused across every model and every test query.

In [ ]:
train_df_for_demos = get_flatten_radqa_data(file_path=TRAIN_FILE_PATH)
demos = sample_demonstrations(train_df_for_demos, k=K_SHOT, seed=DEMO_SEED)
demonstrations_block = render_demonstrations(demos)
print(
    f"K_SHOT={K_SHOT}; demonstrations block length (chars): {len(demonstrations_block)}"
)

In [ ]:
demonstrations_block

In [ ]:
# Run in batches.
# start_idx - end_idx
# 0 - 300
# 300 - 600
# 600 - None

In [ ]:
start_idx = 0
end_idx = 300
wrapper(
    data_subset_file_path=TEST_FILE_PATH,
    vertex_models=config_data["vertex_gcp"],
    result_path=RESULTS_DIR,
    start_idx=start_idx,
    end_idx=end_idx,
    demonstrations_block=demonstrations_block,
)

In [ ]:
start_idx = 300
end_idx = 600
wrapper(
    data_subset_file_path=TEST_FILE_PATH,
    vertex_models=config_data["vertex_gcp"],
    result_path=RESULTS_DIR,
    start_idx=start_idx,
    end_idx=end_idx,
    demonstrations_block=demonstrations_block,
)

In [ ]:
start_idx = 600
end_idx = None
wrapper(
    data_subset_file_path=TEST_FILE_PATH,
    vertex_models=config_data["vertex_gcp"],
    result_path=RESULTS_DIR,
    start_idx=start_idx,
    end_idx=end_idx,
    demonstrations_block=demonstrations_block,
)